In [0]:
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from typing import Dict, Union, List, Optional

def merge_child_to_parent_dim(
    spark,
    child_table,
    parent_table_name: str,
    column_mapping: Dict[str, Union[str, F.Column]],
    merge_key: Union[str, List[str]],
    update_set: Optional[Dict[str, str]] = None,
    insert_values: Optional[Dict[str, str]] = None
) -> None:
    """
    Executes a Delta Lake SCD Type 1 (Upsert) MERGE from a child table to a parent dimension table.
    """
    # 1. Reference target parent table
    parent_table = DeltaTable.forName(spark, parent_table_name)

    # 2. Prepare child table source aligned with schema requirements
    select_exprs = [
        (F.col(src).alias(tgt) if isinstance(src, str) else src.alias(tgt))
        for src, tgt in column_mapping.items()
    ]
    child_table_df = child_table.select(*select_exprs)

    # 3. Construct dynamic join condition (Supports single string or list of keys)
    if isinstance(merge_key, str):
        merge_keys = [merge_key]
    else:
        merge_keys = merge_key

    join_condition = " AND ".join([f"target.{key} = source.{key}" for key in merge_keys])

    # 4. Initialize Delta MERGE builder
    merge_builder = (
        parent_table.alias("target")
        .merge(child_table_df.alias("source"), join_condition)
    )

    # 5. Handle Update clause
    if update_set:
        merge_builder = merge_builder.whenMatchedUpdate(set=update_set)
    else:
        merge_builder = merge_builder.whenMatchedUpdateAll()

    # 6. Handle Insert clause
    if insert_values:
        merge_builder = merge_builder.whenNotMatchedInsert(values=insert_values)
    else:
        merge_builder = merge_builder.whenNotMatchedInsertAll()

    # 7. Execute transaction
    merge_builder.execute()